# Setup Silver tables

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType, DecimalType, DataType
from utils.logger import get_logger
from etl_config.silver_config import SILVER_CONFIG

logger = get_logger("silver_setup")

In [0]:
def get_spark_type(type_str: str) -> DataType:
    """Convert string type to Spark DataType."""
    type_map = {
        "integer": IntegerType(),
        "string": StringType(),
        "date": DateType(),
        "timestamp": TimestampType(),
        "decimal(10,2)": DecimalType(10, 2),
        "decimal(10,4)": DecimalType(10, 4),
        "decimal(3,2)": DecimalType(3, 2),
    }
    return type_map.get(type_str, StringType())

In [0]:
def create_silver_schema(cfg) -> StructType:
    """Create Spark schema from silver config."""
    fields = []
    
    for col_name, col_type in cfg.cast_columns.items():
        spark_type = get_spark_type(col_type)
        nullable = col_name not in cfg.required_columns
        fields.append(StructField(col_name, spark_type, nullable))
    
    return StructType(fields)

In [0]:
# Create silver tables
for table_key, cfg in SILVER_CONFIG.items():
    table_name = cfg.target_table
    
    if spark.catalog.tableExists(table_name):
        logger.info(f"Table already exists: {table_name}")
        continue
    
    logger.info(f"Creating silver table: {table_name}")
    
    # Create schema
    schema = create_silver_schema(cfg)
    
    # Create empty table
    df = spark.createDataFrame([], schema)
    df.write.format("delta").saveAsTable(table_name)
    
    # Set properties
    spark.sql(
        f"""
            ALTER TABLE {table_name}
            SET TBLPROPERTIES (
                'delta.autoOptimize.optimizeWrite' = 'true',
                'delta.autoOptimize.autoCompact' = 'true',
                'description' = :table_description
        )""",
        args={"table_description": cfg.table_description}
    )
    
    # Add column comments
    for column, comment in cfg.column_comments.items():
        try:
            spark.sql(
                f"""
                    COMMENT ON COLUMN {table_name}.{column}
                    IS :comment
                """,
                args={"comment": comment}
            )
            logger.info(f"Added comment to column: {table_name}.{column}")
        except Exception as e:
            logger.warning(f"Could not add comment for {column}: {e}")
    
    logger.info(f"Created: {table_name}")

logger.info("Silver table setup complete.")

### Watermark table

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}._watermark (
        table_name   STRING    NOT NULL,
        batch_id     STRING    NOT NULL,
        ingested_at  TIMESTAMP,
        processed_at TIMESTAMP,
        rows_merged  LONG,
        status       STRING    NOT NULL
    )
    USING DELTA
    COMMENT 'Tracking of processed Bronze batches per table'
""")
logger.info(f"Watermark table created/verified: {SILVER}._watermark")

spark.sql(f"""
    ALTER TABLE {SILVER}._watermark
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")
logger.info(f"Table properties applied for: {SILVER}._watermark")

In [0]:
# quarantine table
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}._quarantine (
        table_name       STRING    NOT NULL,
        batch_id         STRING,
        rejected_at      TIMESTAMP,
        error_type       STRING    NOT NULL,
        _missing_columns ARRAY<STRING>,
        raw_data         STRING
    )
    USING DELTA
    COMMENT 'Bad rows due to missing or uncastable values'
""")
logger.info(f"Quarantine table created/verified: {SILVER}._quarantine")

spark.sql(f"""
    ALTER TABLE {SILVER}._quarantine
    SET TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")
logger.info(f"Table properties applied for: {SILVER}._quarantine")